In [1]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
import requests

In [2]:
# tools create
@tool
def multiply(a: int, b: int)  ->int:
    """Give 2 number a and b this tool returns their product"""
    return a * b

In [3]:
print(multiply.invoke({'a':2,'b':4}))

8


#### Tool binding

In [4]:
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()
llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0
)


In [5]:
llm_with_tools = llm.bind_tools([multiply])

In [6]:
llm_with_tools.invoke('hi how are you')

AIMessage(content="Hello! I'm just a bunch of code, so I don't have feelings, but I'm here and ready to help you. How can I assist you today?", additional_kwargs={'reasoning_content': 'User says "hi how are you". We respond politely.'}, response_metadata={'token_usage': {'completion_tokens': 53, 'prompt_tokens': 133, 'total_tokens': 186, 'completion_time': 0.070641928, 'completion_tokens_details': {'reasoning_tokens': 13}, 'prompt_time': 0.007656599, 'prompt_tokens_details': None, 'queue_time': 0.341863463, 'total_time': 0.078298527}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_34cb00738d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10ce6-c5e5-72f1-a491-22e86fe9b5a8-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 133, 'output_tokens': 53, 'total_tokens': 186, 'output_token_details': {'reasoning': 13}})

#### Tool calling -->  is the process where llm (language model) decides, during a conversation or task. that it needs to use a specific tool (function) -- and generates a structured output with
* the name of the tools
* and the arguments to call it with

In [7]:
llm_with_tools.invoke("can you multiply 5 with 23").tool_calls[0] # llm can not call itself tools and generate answer insted it suggest you to use this tools

{'name': 'multiply',
 'args': {'a': 5, 'b': 23},
 'id': 'fc_6b18dedb-fcfb-4a48-92d4-c5e1c4e6d333',
 'type': 'tool_call'}

#### Tool Exrcution --> Tool execution is the step where the actual python function (tool) is run using the imput arguments the the llm suggested during tool calling

In [8]:
query = HumanMessage('can you multiply 4 with 8')

In [9]:
messages = [query]

In [10]:
result = llm_with_tools.invoke(messages)

In [11]:
result

AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_e546ce4c-8823-4e86-a7c3-c1af760bdcad', 'function': {'arguments': '{"a":4,"b":8}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 137, 'total_tokens': 172, 'completion_time': 0.039087236, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009250286, 'prompt_tokens_details': None, 'queue_time': 0.423431561, 'total_time': 0.048337522}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10ce6-cb7e-7942-91e1-9565a46b60c0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 8}, 'id': 'fc_e546ce4c-8823-4e86-a7c3-c1af760bdcad', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 137, 'output_tokens': 3

In [12]:
messages.append(result)

In [13]:
messages

[HumanMessage(content='can you multiply 4 with 8', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_e546ce4c-8823-4e86-a7c3-c1af760bdcad', 'function': {'arguments': '{"a":4,"b":8}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 137, 'total_tokens': 172, 'completion_time': 0.039087236, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009250286, 'prompt_tokens_details': None, 'queue_time': 0.423431561, 'total_time': 0.048337522}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10ce6-cb7e-7942-91e1-9565a46b60c0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 8}, 'id': 'fc_e546ce4c-8823-4e86-a7c3-c1af760bdcad', 'typ

In [14]:
result.tool_calls[0]['args']

{'a': 4, 'b': 8}

In [15]:
tool_result = multiply.invoke(result.tool_calls[0])

In [16]:
messages.append(tool_result)

In [17]:
messages

[HumanMessage(content='can you multiply 4 with 8', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'reasoning_content': 'We need to use the multiply function.', 'tool_calls': [{'id': 'fc_e546ce4c-8823-4e86-a7c3-c1af760bdcad', 'function': {'arguments': '{"a":4,"b":8}', 'name': 'multiply'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 35, 'prompt_tokens': 137, 'total_tokens': 172, 'completion_time': 0.039087236, 'completion_tokens_details': {'reasoning_tokens': 9}, 'prompt_time': 0.009250286, 'prompt_tokens_details': None, 'queue_time': 0.423431561, 'total_time': 0.048337522}, 'model_name': 'openai/gpt-oss-20b', 'system_fingerprint': 'fp_565badff47', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a10ce6-cb7e-7942-91e1-9565a46b60c0-0', tool_calls=[{'name': 'multiply', 'args': {'a': 4, 'b': 8}, 'id': 'fc_e546ce4c-8823-4e86-a7c3-c1af760bdcad', 'typ

In [18]:
llm_with_tools.invoke(messages).content

'Sure! 4 × 8 = **32**.'